# Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7cd7f3778c20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7cd7f3779940>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating of the movie out of 10")

In [7]:
structured_model = model.with_structured_output(Movie)
structured_model

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7f32f6121d10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f32f6122710>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The year the movie was released', 'type': 'integer'}, 'rating': {'description': 'The rating of the movie out of 10', 'type': 'number'}}, 'required': ['title',

In [9]:
model.invoke("Provide details about the movie Inception").content

'**Inception (2010) – Quick Reference Guide**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Year** | 2010 |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Running Time** | 148 minutes (≈2\u202fh\u202f28\u202fmin) |\n| **MPAA Rating** | PG‑13 (Violence, Some language, Brief strong sexual content) |\n| **Country** | United States |\n| **Language** | English (with some French, Japanese, and other languages in background dialogue) |\n| **Budget** | Approximately **$160\u202fmillion** |\n| **Box‑Office Gross** | ≈ **$836\u202fmillion** worldwide (one of the highest‑grossing films of 2010) |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Distributor** | Warner Bros. Pictures |\n\n---\n\n### Key Creative Personnel\n| Role | Name |\n|------|------|\n| **Director** | Christopher Nolan |\n| **Writer(s)** | Christopher Nolan (original screenplay) |\n| **Producer(s)** | Emma Thomas, Christopher Nola

In [10]:
structured_response = structured_model.invoke("Provide details about the movie Inception")
structured_response

Movie(title='Inception', director='Christopher Nolan', release_year=2010, rating=8.8)

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  # include_raw=True to get the raw response along with the structured output

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about the movie Inception". We have a function to get movie details: functions.Movie with director, rating, title, year. We need to call it with appropriate parameters. Provide director: Christopher Nolan, rating maybe 8.8 (IMDb). Title: Inception. Year: 2010.\n\nWe need to call function.', 'tool_calls': [{'id': 'fc_8c2c16a9-3f04-47c1-a881-5ebc04677a14', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 164, 'total_tokens': 291, 'completion_time': 0.268238344, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.0066519, 'prompt_tokens_details': None, 'queue_time': 0.04548137, 'total_time': 0.274890244}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier'

### Nested Structure

In [6]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class Movie(BaseModel):
    title: str
    year: int
    director: str
    rating: float
    actors: list[Actor]
    genre: list[str]

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genre=['Action', 'Sci-Fi', 'Thriller'])

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [5]:
from typing_extensions import TypedDict, Annotated

class MovieDetail(TypedDict):
    name: Annotated[str, ..., "The title of the movie"]   
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]
    budget: Annotated[float, ..., "The budget of the movie in millions"]

model_with_structure = model.with_structured_output(MovieDetail)
response = model_with_structure.invoke("Provide details about the movie Avengers: Endgame")
response

{'budget': 356,
 'director': 'Anthony Russo',
 'name': 'Avengers: Endgame',
 'rating': 8.4,
 'year': 2019}

In [9]:
class MovieWithActors(TypedDict):
    name: Annotated[str, ..., "The title of the movie"]   
    role: Annotated[str, ..., "role of the actor in the movie"]

class MovieDetailWithCast(TypedDict):
    name: Annotated[str, ..., "The title of the movie"]   
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    actors: list[MovieWithActors]
    genre: list[str]

model_with_structure = model.with_structured_output(MovieDetailWithCast)
response = model_with_structure.invoke("Provide full details about the movie Avengers: Endgame")
response
    

{'actors': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Don Cheadle', 'role': 'James Rhodes / War Machine'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Brie Larson', 'role': 'Carol Danvers / Captain Marvel'},
  {'name': 'Karen Gillan', 'role': 'Nebula'},
  {'name': 'Danai Gurira', 'role': 'Okoye'},
  {'name': 'Benedict Wong', 'role': 'Wong'},
  {'name': 'Jon Favreau', 'role': 'Harold "Happy" Hogan'},
  {'name': 'Josh Brolin', 'role': 'Thanos'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Tom Holland', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Chris Pratt', 'role': 'Peter Quill / Star-Lord (c

In [10]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
# Pydentic models can be used to define the expected structure of the output from the model. This allows for better validation and parsing of the response, ensuring that the data is in the correct format and contains all necessary information.
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict 
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"] # response is a dict with keys "name", "email", and "phone"

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [18]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')